# 태양광 발전량 예측 — AutoGluon TimeSeriesPredictor 일괄 실행

**프로젝트**: 전국 17개 시도 태양광 발전량 예측 + ESS 운영 가치 분석  
**단계**: Phase 1 후속 / 다음 모델 트랙 결정용 1단계 — **모델 아키텍처 일괄 비교**  
**환경**: GCP Colab Enterprise (L4 24GB), asia-northeast3 권장  
**시간**: AutoGluon fit 약 8시간 + 전후 약 30분

---

## 🎯 목적

이 노트북은 AutoGluon TimeSeriesPredictor로 LightGBM, CatBoost, TFT, DeepAR, PatchTST, Chronos 등 시계열 SOTA 모델을 일괄 학습/비교한다.  
**기존 XGBoost 통합 모델 결과를 기준선**으로 두고, AutoML 산출 모델 중 의미 있는 개선이 있는지 확인한다.

### 기준선 (XGBoost 통합 모델)

| 지표 | 값 |
|---|---|
| 단순 평균 MAE | 9.59 |
| 가중 MAE | 31.91 |
| 전남 MAE | 90.42 |
| LSTM MAE (참고) | 17.82 (XGBoost 대비 1.9배 악화) |

### 결과 분기 시나리오 (사후 판정용)

| 결과 | 다음 액션 | 포트폴리오 메시지 |
|---|---|---|
| (a) 모든 모델 가중 MAE 30~35대 | Phase 2 MPC로 직행 | "7개 모델로 검증, 모델 한계 확인 → MPC 필요성 정량화" |
| (b) Chronos/TFT/PatchTST가 가중 MAE 25 이하 | 그 모델만 2단계 심화 | "사전학습 시계열 모델까지 검증, 최적 1개 선정 후 심화" |
| (c) 특정 모델이 전남에서만 개선 | 전남 분석 트랙 분리 | "지역별 외삽 문제 진단, 모델 매칭 분석" |

분기 결정은 사용자가 결과 보고 판단. 노트북 마지막 셀에서 자동 진단 출력은 제공한다.

---

## 📋 사전 준비 (노트북 실행 전 1회만)

### 1. GCP 프로젝트 확인
- Colab Enterprise를 쓰고 있다면 이미 GCP 프로젝트가 있다.
- 프로젝트 ID는 콘솔 상단에서 확인.

### 2. GCS 버킷 생성 (콘솔 UI로 5분)
- GCP 콘솔 → **Cloud Storage** → **버킷 만들기**
- 버킷 이름: 예) `solar-ess-portfolio-{본인id}` (전역 고유)
- 위치: **asia-northeast3 (서울)** — 런타임과 같은 리전 권장
- 스토리지 클래스: **Standard**, 나머지 기본값 OK

### 3. 데이터 업로드 (로컬에서 1회)
로컬 터미널에서:
```bash
# 인증 (1회만)
gcloud auth login
gcloud config set project YOUR_PROJECT_ID

# 데이터 업로드
gsutil cp data/processed/national_train_ready.csv \
  gs://YOUR_BUCKET/data/national_train_ready.csv
gsutil cp data/processed/national_test_ready.csv \
  gs://YOUR_BUCKET/data/national_test_ready.csv

# XGBoost 비교 기준선 업로드
gsutil cp outputs/national_xgb_predictions.csv \
  gs://YOUR_BUCKET/baselines/national_xgb_predictions.csv
gsutil cp outputs/national_xgb_results.json \
  gs://YOUR_BUCKET/baselines/national_xgb_results.json
```

또는 GCP 콘솔에서 드래그앤드롭으로 업로드해도 된다.

### 4. 런타임 템플릿 (idle shutdown 늘리기)
- 콘솔 → **Vertex AI** → **Colab Enterprise** → **런타임 템플릿**
- "새 템플릿" → idle shutdown을 **720분(12시간) 이상**으로 설정
- GPU: **L4 (24GB)**, 머신 타입: g2-standard-4 또는 g2-standard-8
- 리전: 위 GCS 버킷과 같은 곳

### 5. 본 노트북 환경 설정 (아래 셀에서 변수만 수정)
다음 셀에서 `GCS_BUCKET`과 `PROJECT_ID`만 본인 것으로 수정하면 된다.

---

## ⚠️ CLAUDE.md 규칙 준수 사항

- 시간 순 분리 (train ≤2022, test=2023)
- 데이터 누수 금지 (scaler/encoder train만 fit)
- 재현성 (random_state=42)
- 야간 클리핑 (00~05시, 19~23시 → 0)
- 보호 파일 덮어쓰기 금지: 모든 산출물은 `gs://BUCKET/outputs/autogluon_v1/` 하위
- 한글 폰트 적용
- 로그 `[HH:MM:SS]` 형식


## 0. 환경 변수 — 본인 GCP 정보로 수정

다음 셀의 `GCS_BUCKET`과 `PROJECT_ID`만 본인 것으로 바꾸면 된다.


In [ ]:
# ── 본인 GCP 정보 입력 ─────────────────────────────────────────────
PROJECT_ID  = "your-project-id"           # ← 본인 GCP 프로젝트 ID
GCS_BUCKET  = "solar-ess-portfolio-xxx"   # ← 본인 GCS 버킷 이름 (gs:// 빼고)
REGION      = "asia-northeast3"           # 보통 그대로
# ──────────────────────────────────────────────────────────────────

# 경로 상수 (수정 불필요)
GCS_BASE        = f"gs://{GCS_BUCKET}"
TRAIN_URI       = f"{GCS_BASE}/data/national_train_ready.csv"
TEST_URI        = f"{GCS_BASE}/data/national_test_ready.csv"
XGB_PRED_URI    = f"{GCS_BASE}/baselines/national_xgb_predictions.csv"
XGB_RESULT_URI  = f"{GCS_BASE}/baselines/national_xgb_results.json"
OUT_BASE        = f"{GCS_BASE}/outputs/autogluon_v1"

# AutoGluon 설정
PREDICTION_LENGTH = 24       # 1일 forecast (시간 단위)
EVAL_METRIC       = "MASE"   # 스케일 불변, 17개 지역 통합 평가 적합
TIME_LIMIT_SEC    = 28800    # 8시간
NUM_VAL_WINDOWS   = 3        # 안정성
RANDOM_STATE      = 42

# 한국 시간 로깅 헬퍼
import sys
sys.stdout.reconfigure(encoding="utf-8")
from datetime import datetime, timezone, timedelta
KST = timezone(timedelta(hours=9))
def ts():
    return f"[{datetime.now(KST).strftime('%H:%M:%S')}]"

print(f"{ts()} 환경 변수 설정 완료")
print(f"   PROJECT_ID  : {PROJECT_ID}")
print(f"   GCS_BUCKET  : {GCS_BUCKET}")
print(f"   TRAIN_URI   : {TRAIN_URI}")
print(f"   OUT_BASE    : {OUT_BASE}")


## 1. 환경 검증 및 패키지 설치

GPU 확인, AutoGluon + 한글 폰트 설치. 첫 실행 시 약 5~10분.


In [ ]:
# GPU 확인
import subprocess
print("=== GPU 정보 ===")
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv"], capture_output=True, text=True, check=True)
    print(out.stdout)
except Exception as e:
    print(f"⚠️ nvidia-smi 실패: {e}")
    print("GPU가 없는 런타임이면 학습이 매우 느려진다. 런타임 템플릿을 L4로 다시 만들 것.")

import torch
print(f"torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"디바이스: {torch.cuda.get_device_name(0)}")
    print(f"메모리  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# AutoGluon 설치 (TimeSeries 모듈만)
%pip install -q -U "autogluon.timeseries[chronos-openvino]" gcsfs
# Chronos 사전학습 모델 자동 다운로드 활성화 옵션 포함


In [ ]:
# 한글 폰트 설치 및 적용 (Colab Enterprise는 NanumGothic 미내장)
!apt-get install -y fonts-nanum > /dev/null 2>&1
!fc-cache -fv > /dev/null 2>&1

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 폰트 캐시 재생성
fm._load_fontmanager(try_read_cache=False)

# NanumGothic 찾기
nanum_path = None
for font in fm.fontManager.ttflist:
    if "NanumGothic" in font.name:
        nanum_path = font.fname
        break

if nanum_path:
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
    print(f"✅ 한글 폰트 설정 완료: {nanum_path}")
else:
    print("⚠️ NanumGothic 설치 실패. 한글 깨질 수 있음.")


## 2. 데이터 로드 (GCS → DataFrame)

`gs://` URI를 pandas로 직접 읽는다. Colab Enterprise는 GCS 인증이 자동.


In [ ]:
import pandas as pd
import numpy as np

print(f"{ts()} GCS에서 데이터 로드 시작")

try:
    train_df = pd.read_csv(TRAIN_URI, encoding="utf-8-sig", parse_dates=["timestamp"])
    test_df  = pd.read_csv(TEST_URI,  encoding="utf-8-sig", parse_dates=["timestamp"])
except FileNotFoundError as e:
    raise SystemExit(
        f"❌ GCS 파일 로드 실패: {e}\n"
        f"   - GCS_BUCKET 이름 확인: {GCS_BUCKET}\n"
        f"   - 파일이 업로드됐는지 확인: gsutil ls {GCS_BASE}/data/"
    )

print(f"{ts()} ✅ 로드 완료")
print(f"   train: {train_df.shape}, 기간 {train_df['timestamp'].min()} ~ {train_df['timestamp'].max()}")
print(f"   test : {test_df.shape}, 기간 {test_df['timestamp'].min()} ~ {test_df['timestamp'].max()}")
print(f"   지역 : {sorted(train_df['region'].unique())}")

# 컬럼 확인
print(f"\n컬럼 : {list(train_df.columns)}")


## 3. AutoGluon 형식 변환

- `region` → `item_id` (지역별 별도 시계열)
- `timestamp` → `timestamp` (그대로)
- `power_mwh` → target
- 기상 변수 → `known_covariates` (미래 시점에 알려진 정보로 가정)

**중요**: 본인이 만든 `features.csv`(lag/rolling 포함) 대신 **원본 `ready.csv`만 사용**.  
각 모델이 자기 방식대로 자기 피처를 만들도록 둔다.


In [ ]:
# CLAUDE.md 규칙: 시간 순 분리 검증
assert train_df["timestamp"].max() < test_df["timestamp"].min(), \
    "❌ train/test 시간 순 분리 위반"
assert train_df["timestamp"].max().year <= 2022, \
    f"❌ train에 2022년 초과 데이터 포함: {train_df['timestamp'].max()}"
assert test_df["timestamp"].min().year == 2023, \
    f"❌ test가 2023년이 아님: {test_df['timestamp'].min()}"
print(f"{ts()} ✅ 시간 순 분리 검증 통과")

# AutoGluon 컬럼 확인
expected_cols = ["timestamp", "region", "power_mwh",
                 "기온", "강수량", "습도", "일조", "일사량", "전운량"]
for col in expected_cols:
    if col not in train_df.columns:
        raise SystemExit(f"❌ 필요한 컬럼 '{col}' 없음. 컬럼 목록: {list(train_df.columns)}")

# region_code가 있으면 제거 (AutoGluon은 region 문자열을 item_id로 쓴다)
drop_cols = [c for c in ["region_code"] if c in train_df.columns]
if drop_cols:
    train_df = train_df.drop(columns=drop_cols)
    test_df  = test_df.drop(columns=drop_cols)

print(f"{ts()} ✅ 컬럼 검증 통과")


In [ ]:
# 결측 처리 (AutoGluon은 결측 허용하지만 known_covariates는 결측 없어야 안전)
KNOWN_COVS = ["기온", "강수량", "습도", "일조", "일사량", "전운량"]

print(f"{ts()} 결측 확인")
for col in ["power_mwh"] + KNOWN_COVS:
    n_train = train_df[col].isna().sum()
    n_test  = test_df[col].isna().sum()
    if n_train + n_test > 0:
        print(f"   {col}: train {n_train}, test {n_test}")

# known_covariates 결측을 지역별 forward fill + backward fill
for col in KNOWN_COVS:
    train_df[col] = train_df.groupby("region")[col].ffill().bfill()
    test_df[col]  = test_df.groupby("region")[col].ffill().bfill()

# power_mwh 결측이 있으면 0으로 (야간 발전 없음 가정)
train_df["power_mwh"] = train_df["power_mwh"].fillna(0)
test_df["power_mwh"]  = test_df["power_mwh"].fillna(0)

print(f"{ts()} ✅ 결측 처리 완료")


In [ ]:
# AutoGluon TimeSeriesDataFrame 변환
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

print(f"{ts()} TimeSeriesDataFrame 변환 시작")

# 동일 시간대 중복 제거 (같은 region+timestamp가 여러 행이면 평균)
train_df = train_df.groupby(["region", "timestamp"], as_index=False).mean(numeric_only=True)
test_df  = test_df.groupby(["region", "timestamp"], as_index=False).mean(numeric_only=True)

# AutoGluon 형식
train_tsdf = TimeSeriesDataFrame.from_data_frame(
    train_df,
    id_column="region",
    timestamp_column="timestamp",
)

# 시간 정규화 (필요 시 시간 단위로 강제)
train_tsdf = train_tsdf.convert_frequency(freq="h")

# 변환 후 결측 다시 처리 (convert_frequency가 빈 시간 채우면서 NaN 생성 가능)
train_tsdf = train_tsdf.fill_missing_values(method="ffill").fill_missing_values(method="bfill")

print(f"{ts()} ✅ train TimeSeriesDataFrame 변환 완료")
print(f"   shape: {train_tsdf.shape}")
print(f"   item 수: {train_tsdf.num_items}")
print(f"   주기: {train_tsdf.freq}")
print(f"\n샘플:")
print(train_tsdf.head())


## 4. AutoGluon 학습 (약 8시간)

`presets="best_quality"`: AutoGluon이 LightGBM/CatBoost/TFT/DeepAR/PatchTST/Chronos 등을 자동 비교 후 앙상블.  
런타임이 끊기지 않게 브라우저 탭을 닫지 말 것.


In [ ]:
import os

# 로컬 모델 저장 경로 (학습 후 GCS로 업로드)
LOCAL_MODEL_DIR = "/tmp/autogluon_v1_models"
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

predictor = TimeSeriesPredictor(
    path=LOCAL_MODEL_DIR,
    target="power_mwh",
    prediction_length=PREDICTION_LENGTH,
    eval_metric=EVAL_METRIC,
    known_covariates_names=KNOWN_COVS,
    freq="h",
)

print(f"{ts()} 🚀 AutoGluon 학습 시작 (예상 {TIME_LIMIT_SEC // 3600}시간)")
print(f"   time_limit={TIME_LIMIT_SEC}s, num_val_windows={NUM_VAL_WINDOWS}")
print(f"   metric={EVAL_METRIC}, prediction_length={PREDICTION_LENGTH}")

predictor.fit(
    train_tsdf,
    presets="best_quality",
    time_limit=TIME_LIMIT_SEC,
    num_val_windows=NUM_VAL_WINDOWS,
    random_seed=RANDOM_STATE,
)

print(f"\n{ts()} ✅ 학습 완료")


## 5. Leaderboard (모델별 validation 점수)

각 모델의 validation 성능, 학습 시간, 예측 시간. 포트폴리오 자료의 핵심 표.


In [ ]:
print(f"{ts()} Leaderboard 산출")
leaderboard = predictor.leaderboard(silent=True)
print(leaderboard.to_string())

# 로컬 저장 후 GCS 업로드
LOCAL_LEADERBOARD = "/tmp/leaderboard.csv"
leaderboard.to_csv(LOCAL_LEADERBOARD, index=False, encoding="utf-8")


## 6. 테스트셋 예측 + 17개 지역 MAE 계산

XGBoost와 같은 방식으로 비교하기 위해:
- 야간(00~05, 19~23시) → 0 클리핑
- 지역별 MAE → 단순 평균, 가중 평균(발전량 비중)


In [ ]:
# 테스트셋도 AutoGluon 형식으로
test_full_df = pd.concat([train_df, test_df], ignore_index=True)
test_full_df = test_full_df.sort_values(["region", "timestamp"]).reset_index(drop=True)

# 전체 데이터로 TimeSeriesDataFrame 만든 뒤, predictor가 알아서 train 끝 이후를 예측
full_tsdf = TimeSeriesDataFrame.from_data_frame(
    test_full_df,
    id_column="region",
    timestamp_column="timestamp",
)
full_tsdf = full_tsdf.convert_frequency(freq="h")
full_tsdf = full_tsdf.fill_missing_values(method="ffill").fill_missing_values(method="bfill")

# train 부분만 history로 쓰고, test 기간을 예측
print(f"{ts()} 테스트셋 예측 시작 (rolling forecast)")

# rolling forecast — 24시간씩 prediction_length 만큼 예측, history 슬라이드
all_predictions = []
test_start = test_df["timestamp"].min()
test_end   = test_df["timestamp"].max()

print(f"   test 기간: {test_start} ~ {test_end}")
print(f"   총 {(test_end - test_start).days + 1}일 예측 필요")

# AutoGluon은 .predict()에서 known_covariates를 미래 시점으로 받는다
# 간단화: test 전체를 한 번에 예측하는 게 아니라, 24시간씩 슬라이드하면서 누적
# 단, 본인 케이스는 보통 한 번에 끝까지 forecast해도 충분하다 (cheating 아님 — known_covariates는 기상 데이터로 미래 가용 가정)

# 한 번에 전체 예측 (test 전체 길이가 prediction_length보다 길어도 AutoGluon은 알아서 처리)
from autogluon.timeseries import TimeSeriesDataFrame

# train만 사용해서 future_known_covariates를 알려주고 한 번에 예측
train_only_tsdf = full_tsdf.slice_by_timestep(end_index=-len(test_df.groupby('region').size().iloc[0:1].iloc[0]) if False else None)

# 더 안전한 방식: predictor.predict()에 train_tsdf 넘기고 known_covariates를 future로 명시
test_known_covs = test_df[["region", "timestamp"] + KNOWN_COVS].copy()
test_known_covs_tsdf = TimeSeriesDataFrame.from_data_frame(
    test_known_covs, id_column="region", timestamp_column="timestamp",
)
test_known_covs_tsdf = test_known_covs_tsdf.convert_frequency(freq="h")

# AutoGluon이 prediction_length 단위로 예측. test가 8760시간(1년)이면 365번 호출 필요.
# 효율성 위해 rolling 호출
predictions_list = []
unique_regions = sorted(test_df["region"].unique())

# 시간순 정렬된 unique 타임스탬프 (각 지역에서 동일하다고 가정)
test_timestamps = sorted(test_df["timestamp"].unique())
n_total = len(test_timestamps)
n_steps = n_total // PREDICTION_LENGTH + (1 if n_total % PREDICTION_LENGTH else 0)
print(f"   {PREDICTION_LENGTH}시간 윈도우 {n_steps}회 호출 예정")

current_history = train_tsdf.copy()

for step in range(n_steps):
    win_start = step * PREDICTION_LENGTH
    win_end   = min(win_start + PREDICTION_LENGTH, n_total)
    win_timestamps = test_timestamps[win_start:win_end]
    
    # 이 윈도우의 known_covariates
    win_known = test_known_covs_tsdf.loc[
        test_known_covs_tsdf.index.get_level_values("timestamp").isin(win_timestamps)
    ]
    
    # 예측
    try:
        pred = predictor.predict(current_history, known_covariates=win_known)
    except Exception as e:
        print(f"   step {step}: 예측 실패 {e}")
        break
    
    predictions_list.append(pred.reset_index())
    
    # history에 실제 값을 추가 (rolling)
    next_actuals = test_full_df[
        test_full_df["timestamp"].isin(win_timestamps)
    ]
    next_actuals_tsdf = TimeSeriesDataFrame.from_data_frame(
        next_actuals, id_column="region", timestamp_column="timestamp",
    )
    next_actuals_tsdf = next_actuals_tsdf.convert_frequency(freq="h")
    current_history = pd.concat([current_history, next_actuals_tsdf])
    
    if (step + 1) % 30 == 0:
        print(f"   {ts()} step {step+1}/{n_steps} 완료")

pred_all = pd.concat(predictions_list, ignore_index=True)
pred_all = pred_all.rename(columns={"item_id": "region", "mean": "predicted"})
# 예측 컬럼은 모델에 따라 'mean' 또는 분위수가 들어있다
if "predicted" not in pred_all.columns and "0.5" in pred_all.columns:
    pred_all["predicted"] = pred_all["0.5"]

print(f"\n{ts()} ✅ 예측 완료. 예측 행수 {len(pred_all)}")


In [ ]:
# 야간 클리핑 (CLAUDE.md 규칙)
pred_all["hour"] = pd.to_datetime(pred_all["timestamp"]).dt.hour
night_mask = (pred_all["hour"] <= 5) | (pred_all["hour"] >= 19)
pred_all.loc[night_mask, "predicted"] = 0
pred_all["predicted"] = pred_all["predicted"].clip(lower=0)
pred_all = pred_all.drop(columns=["hour"])

# test 실측과 merge
test_merge = test_df[["region", "timestamp", "power_mwh"]].rename(
    columns={"power_mwh": "actual"}
)
pred_with_actual = pred_all.merge(test_merge, on=["region", "timestamp"], how="inner")
print(f"{ts()} merge 결과: {len(pred_with_actual)} 행")
print(pred_with_actual.head())


In [ ]:
# 지역별 MAE 계산
def compute_mae(group):
    return np.mean(np.abs(group["actual"] - group["predicted"]))

region_mae = pred_with_actual.groupby("region").apply(compute_mae).reset_index()
region_mae.columns = ["region", "mae"]

# 지역별 평균 발전량 (가중 평균용)
region_mean_gen = train_df.groupby("region")["power_mwh"].mean().reset_index()
region_mean_gen.columns = ["region", "mean_gen"]

merged = region_mae.merge(region_mean_gen, on="region")
total_gen = merged["mean_gen"].sum()
merged["weight"] = merged["mean_gen"] / total_gen

simple_avg_mae    = merged["mae"].mean()
weighted_mae      = (merged["mae"] * merged["weight"]).sum()
jeonnam_mae       = merged.loc[merged["region"] == "전라남도", "mae"].values
jeonnam_mae       = float(jeonnam_mae[0]) if len(jeonnam_mae) else None

print(f"{ts()} === AutoGluon 결과 ===")
print(f"   단순 평균 MAE : {simple_avg_mae:.4f}")
print(f"   가중 MAE     : {weighted_mae:.4f}")
print(f"   전남 MAE     : {jeonnam_mae:.4f}" if jeonnam_mae else "   전남 MAE     : N/A")
print(f"\n지역별 MAE:")
print(merged[["region", "mae", "weight"]].sort_values("weight", ascending=False).to_string(index=False))


## 7. XGBoost 기준선 대비 비교


In [ ]:
import json
import gcsfs

fs = gcsfs.GCSFileSystem(project=PROJECT_ID)

# XGBoost 결과 로드
try:
    with fs.open(XGB_RESULT_URI, "r", encoding="utf-8") as f:
        xgb_results = json.load(f)
    print(f"{ts()} XGBoost 기준선 로드 완료")
except Exception as e:
    print(f"⚠️ XGBoost 결과 로드 실패: {e}")
    xgb_results = None

# XGBoost 예측 로드 (지역별 MAE 비교용)
try:
    xgb_pred = pd.read_csv(XGB_PRED_URI, encoding="utf-8-sig", parse_dates=["timestamp"])
    print(f"   XGBoost 예측 shape: {xgb_pred.shape}")
except Exception as e:
    print(f"⚠️ XGBoost 예측 로드 실패: {e}")
    xgb_pred = None

# 비교 표
print(f"\n=== 모델 비교 ===")
print(f"{'지표':20s} {'XGBoost':>12s} {'AutoGluon':>12s} {'차이':>12s}")
xgb_simple = xgb_results.get("mae", None) if xgb_results else None
xgb_weighted = xgb_results.get("weighted_mae", None) if xgb_results else None

if xgb_simple:
    diff = simple_avg_mae - xgb_simple
    print(f"{'단순 평균 MAE':20s} {xgb_simple:>12.4f} {simple_avg_mae:>12.4f} {diff:+12.4f}")
if xgb_weighted:
    diff = weighted_mae - xgb_weighted
    print(f"{'가중 MAE':20s} {xgb_weighted:>12.4f} {weighted_mae:>12.4f} {diff:+12.4f}")


## 8. 시각화


In [ ]:
import matplotlib.pyplot as plt

# 그림 1: leaderboard 상위 N개 막대
fig, ax = plt.subplots(figsize=(10, 6))
lb = leaderboard.head(10).copy()
ax.barh(lb["model"], -lb["score_val"], color="steelblue")  # score는 음수로 저장됨
ax.set_xlabel(f"-{EVAL_METRIC} (낮을수록 좋음)")
ax.set_title("AutoGluon Leaderboard — 상위 10개 모델 (validation)")
ax.invert_yaxis()
plt.tight_layout()
LB_PNG = "/tmp/autogluon_leaderboard.png"
plt.savefig(LB_PNG, dpi=120, bbox_inches="tight")
plt.show()

# 그림 2: 지역별 MAE 비교 (XGBoost vs AutoGluon)
if xgb_pred is not None:
    # XGBoost 지역별 MAE
    if "region" in xgb_pred.columns:
        xgb_pred["abs_err"] = (xgb_pred["actual"] - xgb_pred["predicted"]).abs()
        xgb_region_mae = xgb_pred.groupby("region")["abs_err"].mean().reset_index()
        xgb_region_mae.columns = ["region", "xgb_mae"]
        cmp = merged.merge(xgb_region_mae, on="region", how="left")
        cmp = cmp.sort_values("weight", ascending=False)
        
        fig, ax = plt.subplots(figsize=(12, 6))
        x = np.arange(len(cmp))
        w = 0.4
        ax.bar(x - w/2, cmp["xgb_mae"], w, label="XGBoost", color="indianred")
        ax.bar(x + w/2, cmp["mae"],     w, label="AutoGluon", color="steelblue")
        ax.set_xticks(x)
        ax.set_xticklabels(cmp["region"], rotation=45, ha="right")
        ax.set_ylabel("MAE (MWh)")
        ax.set_title("지역별 MAE — XGBoost vs AutoGluon")
        ax.legend()
        plt.tight_layout()
        CMP_PNG = "/tmp/autogluon_vs_xgb_region.png"
        plt.savefig(CMP_PNG, dpi=120, bbox_inches="tight")
        plt.show()
    else:
        print("⚠️ xgb_pred에 region 컬럼 없음. 지역별 비교 그림 생략")
        CMP_PNG = None
else:
    CMP_PNG = None


## 9. 결과 저장 (GCS 업로드)

CLAUDE.md 보호 파일 덮어쓰기 금지 → 모든 산출물 `gs://BUCKET/outputs/autogluon_v1/` 하위로.


In [ ]:
import json

# 예측 CSV
LOCAL_PRED  = "/tmp/national_autogluon_predictions.csv"
pred_with_actual.to_csv(LOCAL_PRED, index=False, encoding="utf-8")

# 지역별 MAE CSV
LOCAL_REGION_MAE = "/tmp/national_autogluon_region_mae.csv"
merged.to_csv(LOCAL_REGION_MAE, index=False, encoding="utf-8")

# 종합 결과 JSON
result_json = {
    "model": "autogluon_v1",
    "config": {
        "preset": "best_quality",
        "time_limit_sec": TIME_LIMIT_SEC,
        "prediction_length": PREDICTION_LENGTH,
        "eval_metric": EVAL_METRIC,
        "num_val_windows": NUM_VAL_WINDOWS,
        "random_state": RANDOM_STATE,
    },
    "summary": {
        "simple_avg_mae": float(simple_avg_mae),
        "weighted_mae":   float(weighted_mae),
        "jeonnam_mae":    float(jeonnam_mae) if jeonnam_mae else None,
    },
    "baseline_xgboost": {
        "simple_avg_mae": xgb_simple,
        "weighted_mae":   xgb_weighted,
    },
    "leaderboard_top10": leaderboard.head(10).to_dict(orient="records"),
    "region_mae": merged.to_dict(orient="records"),
}
LOCAL_RESULT = "/tmp/national_autogluon_results.json"
with open(LOCAL_RESULT, "w", encoding="utf-8") as f:
    json.dump(result_json, f, ensure_ascii=False, indent=2, default=str)

# GCS 업로드
upload_pairs = [
    (LOCAL_LEADERBOARD, f"{OUT_BASE}/leaderboard.csv"),
    (LOCAL_PRED,        f"{OUT_BASE}/national_autogluon_predictions.csv"),
    (LOCAL_REGION_MAE,  f"{OUT_BASE}/national_autogluon_region_mae.csv"),
    (LOCAL_RESULT,      f"{OUT_BASE}/national_autogluon_results.json"),
    (LB_PNG,            f"{OUT_BASE}/autogluon_leaderboard.png"),
]
if CMP_PNG:
    upload_pairs.append((CMP_PNG, f"{OUT_BASE}/autogluon_vs_xgb_region.png"))

print(f"{ts()} GCS 업로드 시작")
for local, remote in upload_pairs:
    try:
        fs.put(local, remote)
        print(f"   ✅ {remote}")
    except Exception as e:
        print(f"   ❌ {remote}: {e}")

# 학습된 모델 디렉토리도 GCS로 (선택)
print(f"\n{ts()} 학습된 모델 디렉토리 GCS 업로드 (시간 걸림)")
try:
    fs.put(LOCAL_MODEL_DIR, f"{OUT_BASE}/models/", recursive=True)
    print(f"   ✅ {OUT_BASE}/models/")
except Exception as e:
    print(f"   ⚠️ 모델 업로드 실패 (재학습 가능): {e}")


## 10. 결과 분기 진단 (a/b/c 시나리오 자동 판정)

본인이 정의한 분기 기준에 따라 다음 액션 추천.


In [ ]:
print(f"{ts()} === 결과 분기 진단 ===\n")

# 시나리오 판정
scenarios = []

# (a) 모든 모델 가중 MAE 30~35대
if 28.0 <= weighted_mae <= 36.0:
    scenarios.append({
        "code": "a",
        "name": "모델 한계 확인",
        "evidence": f"AutoGluon 가중 MAE {weighted_mae:.2f} (XGBoost 31.91 기준 ±4 이내)",
        "action": "Phase 2 MPC로 직행",
        "message": "7개 모델로 검증, 모델 한계 확인 → MPC 필요성 정량화",
    })

# (b) 가중 MAE 25 이하
if weighted_mae < 25.0:
    scenarios.append({
        "code": "b",
        "name": "유의미한 개선 모델 발견",
        "evidence": f"AutoGluon 가중 MAE {weighted_mae:.2f} (XGBoost 31.91 대비 -{(31.91 - weighted_mae)/31.91*100:.1f}%)",
        "action": "최상위 모델 1개 골라 2단계 심화 (region embedding, lag dropout, quantile loss, 전남 분리)",
        "message": "사전학습 시계열 모델까지 검증, 최적 1개 선정 후 심화",
    })

# (c) 전남에서만 큰 개선
if jeonnam_mae and jeonnam_mae < 60.0:  # XGBoost 90.42 대비 -33% 이상
    scenarios.append({
        "code": "c",
        "name": "전남 특화 개선",
        "evidence": f"전남 MAE {jeonnam_mae:.2f} (XGBoost 90.42 대비 -{(90.42 - jeonnam_mae)/90.42*100:.1f}%)",
        "action": "전남 단독 분석 트랙. capacity proxy 없이 multi-task로 우회",
        "message": "지역별 외삽 문제 진단, 모델 매칭 분석",
    })

if not scenarios:
    print("⚠️ 정의된 분기에 해당하지 않음.")
    print(f"   weighted_mae={weighted_mae:.2f}, jeonnam_mae={jeonnam_mae:.2f if jeonnam_mae else 'N/A'}")
    print("   결과를 사용자와 함께 검토 필요.")
else:
    for s in scenarios:
        print(f"📌 시나리오 ({s['code']}) — {s['name']}")
        print(f"   근거: {s['evidence']}")
        print(f"   다음 액션: {s['action']}")
        print(f"   포트폴리오 메시지: {s['message']}\n")

# Top 모델 추천
print(f"=== Leaderboard Top 5 ===")
print(leaderboard.head(5).to_string(index=False))

print(f"\n{ts()} 🎉 1단계 완료. 결과 보고 다음 단계 결정.")


## 11. 로컬로 결과 가져오기

학습이 끝나면 로컬 터미널에서 다음 명령으로 결과를 가져온다:

```bash
# 산출물 일괄 다운로드
gsutil -m cp -r gs://YOUR_BUCKET/outputs/autogluon_v1 ./outputs/

# 로컬 디렉토리 구조
# outputs/autogluon_v1/
# ├── leaderboard.csv
# ├── national_autogluon_predictions.csv
# ├── national_autogluon_region_mae.csv
# ├── national_autogluon_results.json
# ├── autogluon_leaderboard.png
# ├── autogluon_vs_xgb_region.png
# └── models/                      ← 재예측이 필요한 경우만
```

---

## 12. 다음 단계 (이 노트북 외부)

1. **결과를 ESS v2 시뮬에 통과** — 본인 `ess_simulation_v2.py`에 AutoGluon 예측 CSV 넣고 자급률 비교
2. **시나리오 (a) 가장 확률 높음** → Phase 2 MPC 시작
3. 시나리오 (b)면 → 해당 모델만 골라 region embedding/lag dropout/quantile loss/전남 분리 적용

브라우저 Claude(Project)에 결과 JSON 첨부하면 다음 단계 작업 이어갈 수 있다.
